# Required Libraries

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import math
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

: 

# EDA: Weather 

## Preprocessing Weather: Load in Data

In [ ]:
import pandas as pd

weather_cols = ["YYYYMMDD", "HH", "DD", "FH", "FF", "FX", "T", "TD", "SQ", "Q", "DR", "RH", "P", "VV", "N", "U", "WW", "IX"]
weather_dtypes = {
    "HH": "float32",
    "DD": "float32",
    "FH": "float32",
    "FF": "float32",
    "FX": "float32",
    "T": "float32",
    "TD": "float32",
    "SQ": "float32",
    "Q": "float32",
    "DR": "float32",
    "RH": "float32",
    "P": "float32",
    "VV": "float32",
    "N": "float32",
    "U": "float32",
    "WW": "float32",
    "IX": "float32",
}

files_and_ranges = [
    ("I:/Weather 2019-2025/uurgeg_260_2011-2020.txt", "2019-01-01", "2020-12-31"),
    ("I:/Weather 2019-2025/uurgeg_260_2021-2030.txt", "2021-01-01", "2025-12-31"),
]

weather = pd.concat(
    [
        pd.read_csv(
            path,
            sep=",",
            skiprows=30,
            usecols=weather_cols,
            dtype=weather_dtypes,
            parse_dates=["YYYYMMDD"],
            low_memory=False
        ).loc[lambda d: d["YYYYMMDD"].between(start, end)]
        for path, start, end in files_and_ranges
    ],
    ignore_index=True
)

In [ ]:
weather.columns = weather.columns.str.strip().str.lstrip("#").str.strip()

weather = (
    weather.rename(columns={
        "STN": "Station",
        "YYYYMMDD": "Date",
        "HH": "Hour",
        "DD": "Wind Direction",
        "FH": "Hourly Mean Wind Speed",
        "FF": "Wind Speed last 10 Minutes",
        "FX": "Max Wind Speed",
        "T": "Temperature",
        "T10N": "Min Temperature",
        "TD": "Dew Point temperature",
        "SQ": "Sunshine Duration",
        "Q": "Global Radiation",
        "DR": "Precipitation Duration",
        "RH": "Precipitation Amount",
        "P": "Air Pressure",
        "VV": "Horizontal Visibility",
        "N": "Cloud Cover",
        "U": "Humidity",
        "WW": "Weather Code",
        "IX": "Indicator weather code",
        "M": "Fog",
        "R": "Rainfall",
        "S": "Snowfall",
        "O": "Thunder",
        "Y": "Hail",
    })
)

num_cols = weather.columns.drop("Date")
weather[num_cols] = weather[num_cols].apply(pd.to_numeric, errors="coerce", downcast="float")

## Preprocessing Weather: Analyzing the Data

In [ ]:
weather.head()

In [ ]:
# Display basic information about the DataFrame: 
# number of rows, columns, data types, and non-null counts

weather.info()

In [ ]:
weather.describe()

## Preprocessing Weather: Dealing with Missing Values

In [ ]:
# Check for duplicates
weather.duplicated().sum()

In [ ]:
total = weather.isnull().sum().sort_values(ascending=False)
percent = (weather.isnull().sum()/weather.isnull().count()).sort_values(ascending=False)
missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
missing_data

In [ ]:
# Keep missing values 'weather code' and 'Indicator weather code' as it is, since NaN indicates 'No significant weather / not recorded' and is meaningful information for the analysis.
# 'Cloud Cover' has 9% missing values, but I will keep it as it is since NaN

# Drop station because all stations are 260, redundant for analysis. 
weather = weather.drop(columns=['Station', 'min_temperature', 'Weather Code', 'Indicator weather code'])


## Preprocessing Weather: Univariate & Bivariate Analysis

### Preprocessing Weather: Univariate

In [ ]:
weather.describe()

In [ ]:
#boxplot: for outliers and data spread
num_cols = weather.select_dtypes(include=[np.number]).columns
n_cols = 3
n_rows = int(np.ceil(len(num_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(x=weather[col], ax=axes[i])
    axes[i].set_title(f"Boxplot of {col}")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
#histogram: for distribution and central tendency (mean, median)
n = len(num_cols)
n_cols = 3
n_rows = math.ceil(n / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.5 * n_rows))
axes = axes.flatten()

for ax, col in zip(axes, num_cols):
    data = weather[col].dropna()
    sns.histplot(data, bins=30, kde=True, ax=ax, color="skyblue")
    ax.axvline(data.mean(), color="red", linestyle="--", label="Mean")
    ax.axvline(data.median(), color="green", linestyle=":", label="Median")
    ax.set_title(f"Distribution of {col}")
    ax.set_ylabel("Count")
    ax.legend()

for ax in axes[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

### Preprocessing Weather: Bivariate

In [ ]:
#heatmap: for correlation between numeric variables
corr = weather[num_cols].corr()

plt.figure(figsize=(20, 15))
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.5, annot=True, fmt=".2f")
plt.title("Correlation Heatmap of Numeric Weather Variables")
plt.tight_layout()
plt.show()

In [ ]:
#check the top 10 most correlated pairs of numeric variables
corr_abs = weather[num_cols].corr().abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))

top_pairs = upper.stack().sort_values(ascending=False).head(10)
print(top_pairs)

In [ ]:
pairs = top_pairs.head(10).index.tolist()

n_pairs = len(pairs)
n_cols = 2
n_rows = math.ceil(n_pairs / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 4 * n_rows))
axes = np.array(axes).flatten()

for ax, (x, y) in zip(axes, pairs):
    sns.scatterplot(data=weather, x=x, y=y, alpha=0.4, s=20, ax=ax)
    sns.regplot(data=weather, x=x, y=y, scatter=False, color="red", ax=ax)
    ax.set_title(f"{x} vs {y}")

for ax in axes[n_pairs:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

# EDA: Services

## Load in Data and quick Analysis

In [ ]:
#Load in dataset services 2019-2022 (3m8s)
serv19 = pd.read_csv("I:/Datasets Thesis/Train Services 2019-2025/clean/services_clean_2019.csv")
serv20 = pd.read_csv("I:/Datasets Thesis/Train Services 2019-2025/clean/services_clean_2020.csv")
serv21 = pd.read_csv("I:/Datasets Thesis/Train Services 2019-2025/clean/services_clean_2021.csv")
serv22 = pd.read_csv("I:/Datasets Thesis/Train Services 2019-2025/clean/services_clean_2022.csv")


In [ ]:
serv23 = pd.read_csv("I:/Datasets Thesis/Train Services 2019-2025/clean/services_clean_2023.csv")
serv24 = pd.read_csv("I:/Datasets Thesis/Train Services 2019-2025/clean/services_clean_2024.csv")
serv25 = pd.read_csv("I:/Datasets Thesis/Train Services 2019-2025/clean/services_clean_2025.csv")

In [ ]:
#Concatenate dataset services as one (2m45s) 

df_train = pd.concat([serv19, serv20, serv21, serv22, serv23, serv24, serv25], ignore_index= True)
serv_col_names = df_train.columns
print( f" The column names of services are: {serv_col_names}" )

rows, columns = df_train.shape
print( f"df_train has Rows: {rows}, Columns: {columns}" ) #Rows: 38571, Columns: 14


In [ ]:
df_train.info()

In [ ]:
df_train["Service:Company"].value_counts()

In [ ]:
df_train["Service:Type"].value_counts()

## Preprocessing Services: Missing Values

In [ ]:
total_serv19 = df_train.isnull().sum().sort_values(ascending=False)
percent_serv19 = (df_train.isnull().sum()/df_train.isnull().count()).sort_values(ascending=False)
missing_data_serv19 = pd.concat([total_serv19, percent_serv19], axis=1, keys=['Total', 'Percent'])
missing_data_serv19

In [ ]:
# Rows where departure cancellation is missing
departure_null = df_train[df_train["Stop:Departure cancelled"].isna()].copy()

# Missingness flags
departure_null["departure_time_isna"] = departure_null["Stop:Departure time"].isna()
departure_null["departure_delay_isna"] = departure_null["Stop:Departure delay"].isna()

# Overview table
summary_departure_null = pd.crosstab(
    departure_null["departure_time_isna"],
    departure_null["departure_delay_isna"],
    margins=True
)
display(summary_departure_null)

# Explicit rule check: both must be NaN
rule_holds = (departure_null["departure_time_isna"] & departure_null["departure_delay_isna"]).all()
print("Rule holds:", rule_holds)

# Show counterexamples, if any
violations = departure_null[~(departure_null["departure_time_isna"] & departure_null["departure_delay_isna"])]
print("Violations:", len(violations))
if len(violations) > 0:
    display(violations.head(20))

In [ ]:
# Rows where arrival cancellation is missing
arrival_null = df_train[df_train["Stop:Arrival cancelled"].isna()].copy()

# Missingness flags
arrival_null["arrival_time_isna"] = arrival_null["Stop:Arrival time"].isna()
arrival_null["arrival_delay_isna"] = arrival_null["Stop:Arrival delay"].isna()

# Overview table
summary_arrival_null = pd.crosstab(
    arrival_null["arrival_time_isna"],
    arrival_null["arrival_delay_isna"],
    margins=True
)
display(summary_arrival_null)

# Explicit rule check: both must be NaN
rule_holds = (arrival_null["arrival_time_isna"] & arrival_null["arrival_delay_isna"]).all()
print("Rule holds:", rule_holds)

# Show counterexamples, if any
violations = arrival_null[~(arrival_null["arrival_time_isna"] & arrival_null["arrival_delay_isna"])]
print("Violations:", len(violations))
if len(violations) > 0:
    display(violations.head(20))

In [ ]:
# Check rows where planned platform is NaN or False
planned_col = df_train["Stop:Planned platform"]
actual_col = df_train["Stop:Actual platform"]

planned_is_false = planned_col.eq(False) | planned_col.astype(str).str.strip().str.lower().eq("false")
planned_is_na_or_false = planned_col.isna() | planned_is_false

planned_platform_null = df_train.loc[planned_is_na_or_false, ["Stop:Planned platform", "Stop:Actual platform"]].copy()

# Create boolean flags for missingness (without overwriting original columns)
planned_platform_null["planned_is_na"] = planned_platform_null["Stop:Planned platform"].isna()
planned_platform_null["actual_is_na"] = planned_platform_null["Stop:Actual platform"].isna()

# Summary: among rows where planned is NaN or False, is actual also NaN?
summary_planned_platform_null = pd.crosstab(
    planned_platform_null["planned_is_na"],
    planned_platform_null["actual_is_na"],
    margins=True
)

display(summary_planned_platform_null)

# Rows that do NOT satisfy "both planned and actual are NaN"
violations = planned_platform_null[
    ~(planned_platform_null["planned_is_na"] & planned_platform_null["actual_is_na"])
]

print(f"Rows checked (planned NaN or False): {len(planned_platform_null)}")
print(f"Rows where both planned and actual are NaN: {len(planned_platform_null) - len(violations)}")
print(f"Violations: {len(violations)}")

if len(violations) > 0:
    display(violations.head(20))

In [ ]:
arr_cancel_na = df_train["Stop:Arrival cancelled"].isna()
dep_cancel_na = df_train["Stop:Departure cancelled"].isna()

# Explicit non-cancelled events
arr_valid = df_train["Stop:Arrival cancelled"].eq(False)
dep_valid = df_train["Stop:Departure cancelled"].eq(False)

# Start station:
# Arrival cancelled = NaN AND Departure cancelled = False
start_station_mask = arr_cancel_na & dep_valid

# End station:
# Departure cancelled = NaN AND Arrival cancelled = False
end_station_mask = dep_cancel_na & arr_valid

# Total missing values
total_arr_missing = arr_cancel_na.sum()
total_dep_missing = dep_cancel_na.sum()

# Counts matching assumption
start_count = start_station_mask.sum()
end_count = end_station_mask.sum()

# Percentages
start_pct = (start_count / total_arr_missing * 100) if total_arr_missing > 0 else 0
end_pct = (end_count / total_dep_missing * 100) if total_dep_missing > 0 else 0

# Summary table
summary = pd.DataFrame({
    "Missing values": [total_arr_missing, total_dep_missing],
    "Matches assumption": [start_count, end_count],
    "Percentage explained": [start_pct, end_pct]
}, index=[
    "Arrival cancelled NaN = Start stations",
    "Departure cancelled NaN = End stations"
])

display(summary)

# Check impossible overlap
print("Overlap between start and end stations:", 
      (start_station_mask & end_station_mask).sum())

In [ ]:
# Create a subset of the DataFrame with only the cancellation-related columns
cancelled_columns = df_train[[
    "Stop:Arrival cancelled",
    "Stop:Departure cancelled",
    "Service:Partly cancelled",
    "Service:Completely cancelled"
 ]].copy()

# Group by the cancellation columns and count occurrences
cancelled_columns.groupby([
    "Stop:Arrival cancelled",  
    "Stop:Departure cancelled", 
    "Service:Partly cancelled", 
    "Service:Completely cancelled",], 
    dropna=False).size().reset_index(name='count')

In [ ]:
# Create is_cancelled directly in this cell
arr = df_train["Stop:Arrival cancelled"]
dep = df_train["Stop:Departure cancelled"]
complete = df_train["Service:Completely cancelled"] if "Service:Completely cancelled" in df_train.columns else pd.Series(False, index=df_train.index)

df_train["is_cancelled"] = 1
not_cancelled_mask = (
    (complete == False) &
    (
        (arr.isna() & (dep == False)) |
        ((arr == False) & dep.isna()) |
        ((arr == False) & (dep == False))
    )
)
df_train.loc[not_cancelled_mask, "is_cancelled"] = 0

cancelled_columns["is_cancelled"] = df_train["is_cancelled"]

cancelled_columns.groupby([
    "Stop:Arrival cancelled", 
    "Stop:Departure cancelled", 
    "Service:Partly cancelled", 
    "Service:Completely cancelled",
    "is_cancelled"
], dropna=False).size().reset_index(name='count')

## Data Visualization

In [ ]:
# Cancellation Rate Analysis by Year, Month, Day of Week, and Hour
df_time = df_train.copy()
df_time["Service:Date"] = pd.to_datetime(df_time["Service:Date"], errors="coerce")
df_time = df_time.dropna(subset=["Service:Date"])

df_time["year"] = df_time["Service:Date"].dt.year
df_time["month"] = df_time["Service:Date"].dt.month
df_time["weekday"] = df_time["Service:Date"].dt.day_name()

# Extract hour from departure time
if "Stop:Departure time" in df_time.columns:
    dep_time_utc = pd.to_datetime(df_time["Stop:Departure time"], errors="coerce", utc=True)
    df_time["hour"] = dep_time_utc.dt.hour
else:
    df_time["hour"] = pd.NA

# Cancellation rate by year
rate_year = df_time.groupby("year")["is_cancelled"].mean().mul(100).round(2).reset_index(name="cancel_rate_percentage")
print("Cancellation rate by year (%)")
display(rate_year)

plt.figure(figsize=(10, 4))
sns.barplot(data=rate_year, x="year", y="cancel_rate_percentage", color="steelblue")
plt.title("Cancellation Rate by Year")
plt.ylabel("Cancellation Rate (%)")
plt.xlabel("Year")
plt.tight_layout()
plt.show()

# Cancellation rate by month
rate_month = df_time.groupby("month")["is_cancelled"].mean().mul(100).round(2).reset_index(name="cancel_rate_percentage").sort_values("month")
print("\nCancellation rate by month (%)")
display(rate_month)

plt.figure(figsize=(10, 4))
sns.lineplot(data=rate_month, x="month", y="cancel_rate_percentage", marker="o", color="darkgreen")
plt.title("Cancellation Rate by Month")
plt.ylabel("Cancellation Rate (%)")
plt.xlabel("Month")
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()

# Cancellation rate by day of week
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
rate_weekday = df_time.groupby("weekday")["is_cancelled"].mean().mul(100).round(2).reset_index(name="cancel_rate_percentage")
rate_weekday["weekday"] = pd.Categorical(rate_weekday["weekday"], categories=weekday_order, ordered=True)
rate_weekday = rate_weekday.sort_values("weekday")
print("\nCancellation rate by day of week (%)")
display(rate_weekday)

plt.figure(figsize=(10, 4))
sns.barplot(data=rate_weekday, x="weekday", y="cancel_rate_percentage", color="coral")
plt.title("Cancellation Rate by Day of Week")
plt.ylabel("Cancellation Rate (%)")
plt.xlabel("Day of Week")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Cancellation rate by hour
hour_data = df_time.dropna(subset=["hour"])
if len(hour_data) > 0:
    rate_hour = hour_data.groupby("hour")["is_cancelled"].mean().mul(100).round(2).reset_index(name="cancel_rate_percentage")
    print("\nCancellation rate by hour (%)")
    display(rate_hour)

    plt.figure(figsize=(12, 4))
    sns.lineplot(data=rate_hour, x="hour", y="cancel_rate_percentage", marker="o", color="purple")
    plt.title("Cancellation Rate by Hour of Day")
    plt.ylabel("Cancellation Rate (%)")
    plt.xlabel("Hour (24-hour format)")
    plt.xticks(range(0, 24))
    plt.tight_layout()
    plt.show()

In [ ]:

# Build a summary with cancellation at station level, sorted by total cancellations

# Calculate cancellation count and total stops per station
rate_station = (
    df_time.groupby("Stop:Station name")["is_cancelled"]
    .agg(cancel_count="sum", total_stops="count")
    .assign(cancel_rate_pct=lambda x: ((x["cancel_count"] / x["total_stops"]) * 100).round(2))
    .reset_index()
    .sort_values("cancel_count", ascending=False)
)

# Display the top 10 stations by cancellation count
print("Top 10 stations by number of cancellations")
display(rate_station.head(10))

top10 = rate_station.head(10).sort_values("cancel_count", ascending=True)  # ascending for horizontal bar readability

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=top10, y="Stop:Station name", x="cancel_count", color="darkorange", ax=ax)

# Add rate label at the end of each bar
for i, (_, row) in enumerate(top10.iterrows()):
    ax.text(
        row["cancel_count"] + 10,
        i,
        f"{row['cancel_rate_pct']:.2f}% of {int(row['total_stops'])} stops",
        va="center",
        fontsize=9
    )

ax.set_title("Top 10 Stations by Total Cancellation Count")
ax.set_xlabel("Number of Cancellations")
ax.set_ylabel("Station")
plt.tight_layout()
plt.show()


In [ ]:
# Cancellation rate by service type
rate_type = (
df_time.groupby("Service:Type")["is_cancelled"]
.mean()
.mul(100)
.reset_index(name="cancel_rate_pct")
.sort_values("cancel_rate_pct", ascending=False)
)

print("Cancellation rate by service type (%)")
display(rate_type)

plt.figure(figsize=(10, 5))
sns.barplot(data=rate_type, x="Service:Type", y="cancel_rate_pct", color="purple")
plt.title("Cancellation Rate by Service Type")
plt.ylabel("Rate (%)")
plt.xlabel("Service Type")
plt.xticks(rotation=25, ha="right")
plt.show()

In [ ]:
counts = df_time["is_cancelled"].value_counts().sort_index()
total  = len(df_time)

num_not_cancelled = counts.get(0, 0)
num_cancelled     = counts.get(1, 0)
pct_cancelled     = num_cancelled / total * 100
imbalance_ratio   = num_not_cancelled / num_cancelled if num_cancelled > 0 else float("inf")

# Bar chart
class_labels = ["Not cancelled (0)", "Cancelled (1)"]
class_counts  = [num_not_cancelled, num_cancelled]
class_pcts    = [round(100 - pct_cancelled, 2), round(pct_cancelled, 2)]

fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(class_labels, class_counts, color=["steelblue", "tomato"])

for bar, count, pct in zip(bars, class_counts, class_pcts):
    # Show count inside the bar (near top)
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() * 0.97,
        f"{count:,}",
        ha="center", va="top", fontsize=11, color="white", fontweight="bold"
    )
    # Show percentage just above the bar
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total * 0.003,
        f"{pct:.2f}%",
        ha="center", va="bottom", fontsize=11
    )

ax.set_title("Class Distribution of is_cancelled")
ax.set_ylabel("Number of stops")
plt.tight_layout()
plt.show()

# Combine Dataset

In [ ]:
# Add hour column to df_train using departure time, then arrival time as fallback
dep_time = pd.to_datetime(df_train["Stop:Departure time"], errors="coerce", utc=True)
arr_time = pd.to_datetime(df_train["Stop:Arrival time"], errors="coerce", utc=True)

# Prefer departure hour; if missing, use arrival hour
df_train["Hour"] = dep_time.dt.hour.fillna(arr_time.dt.hour).astype("Int64")

# Remap midnight from 0 to 24 so hour range is 1..24
df_train.loc[df_train["Hour"] == 0, "Hour"] = 24

# Forward-fill any remaining NaN hours (e.g., for rows where both departure and arrival times are missing)
df_train['Hour'] = df_train['Hour'].ffill()

# Quick check
print(df_train["Hour"].value_counts(dropna=False).sort_index())

In [ ]:
df_train = df_train.assign(
    YYYYMMDD=pd.to_datetime(df_train["Service:Date"], errors="coerce").dt.normalize(),
    Hour=pd.to_numeric(df_train["Hour"], errors="coerce").astype("Int64"),
)

comb = df_train.merge(
    weather.rename(columns={"Date": "YYYYMMDD"})
           .assign(
               YYYYMMDD=lambda x: pd.to_datetime(x["YYYYMMDD"], errors="coerce").dt.normalize(),
               Hour=lambda x: pd.to_numeric(x["Hour"], errors="coerce").astype("Int64"),
           )
           .sort_values(["YYYYMMDD", "Hour"])   
           .drop_duplicates(["YYYYMMDD", "Hour"]),
    on=["YYYYMMDD", "Hour"],
    how="left",
    suffixes=("", "_weather"),
)

print(comb.shape)
display(comb.head())

## Baseline: Logistic Regression

In [ ]:
target_col = "is_cancelled"
feature_cols = [
    "Hour",
    "Service:Maximum delay",
    "Stop:Arrival delay",
    "Stop:Departure delay",
    "Stop:Platform change",
    "Temperature",
    "Humidity",
    "Air Pressure",
    "Horizontal Visibility",
    "Cloud Cover",
    "Precipitation Amount",
    "Precipitation Duration",
    "Hourly Mean Wind Speed",
    "Max Wind Speed",
    "Fog",
    "Rainfall",
    "Snowfall",
    "Thunder",
    "Hail",
]

X = comb[feature_cols].copy()
X = X.apply(pd.to_numeric, errors="coerce").fillna(0)
y = comb["is_cancelled"]

logist = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="liblinear"
)

logist.fit(X, y)
y_pred = logist.predict(X)

print(classification_report(y, y_pred))